# Notebook 01: Cleaning and Preprocessing ACS County Data

This notebook prepares the county-level dataset used in the project. The main goal is to load multiple American Community Survey (ACS) tables, clean and standardize each one, merge them into a single master dataset, and save the processed output for later analysis.

The cleaned dataset will be used in the EDA and modeling stages of the project.

In [2]:
# -------------------------------------------------------
# Notebook 01: Cleaning & Preprocessing (ACS County Data)
# -------------------------------------------------------
# Purpose:
# - Load ACS county-level source tables
# - Clean and standardize each dataset
# - Merge them into one master dataset
# - Save the final processed file for later analysis
# -------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path

# Display settings for easier inspection
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Create output folder for processed data if it does not already exist
Path("data/processed").mkdir(parents=True, exist_ok=True)

print("Setup complete. Ready to load datasets.")

Setup complete. Ready to load datasets.


## Load Population Dataset (ACS Table B01003)

This section loads the population dataset from the American Community Survey (ACS).

The B01003 table provides total population counts for each U.S. county. This variable is important because it helps provide context for the size of each county and will be used in the final merged dataset.

In [4]:
# -------------------------------------------------------
# Load & Inspect Population Dataset (B01003)
# -------------------------------------------------------

# Load population dataset
population_raw = pd.read_csv("ACSDT5Y2024.B01003-Data.csv")

# Check dataset size
print("Raw dataset shape:", population_raw.shape)

# Preview first few rows
population_raw.head()

Raw dataset shape: (3223, 5)


,GEO_ID,NAME,B01003_001E,B01003_001M,Unnamed: 4
0,Geography,Geographic Area Name,Estimate!!Total,Margin of Error!!Total,NaN
1,0500000US01001,"Autauga County, Alabama",59947,*****,NaN
2,0500000US01003,"Baldwin County, Alabama",246989,*****,NaN
3,0500000US01005,"Barbour County, Alabama",24643,*****,NaN
4,0500000US01007,"Bibb County, Alabama",22130,*****,NaN


## Cleaning Population Dataset

The raw population dataset contains extra metadata rows and columns that are not required for analysis.

In this step, the dataset is cleaned by:
- Removing unnecessary header rows
- Extracting the county FIPS code for merging datasets
- Converting population values to numeric format
- Keeping only relevant columns for analysis

The result is a clean dataset containing county identifiers and total population.

In [6]:
# -------------------------------------------------------
# Clean Population Dataset
# -------------------------------------------------------

# Remove the first row (extra header information)
population_df = population_raw.iloc[1:].copy()

# Extract 5-digit county FIPS code from GEO_ID
population_df["county_fips"] = population_df["GEO_ID"].str[-5:]

# Convert population values to numeric (handle errors as NaN)
population_df["total_population"] = pd.to_numeric(
    population_df["B01003_001E"],
    errors="coerce"
)

# Select relevant columns and rename for clarity
population_clean = population_df[
    ["county_fips", "NAME", "total_population"]
].rename(columns={"NAME": "county_name"})

# Reset index for clean structure
population_clean = population_clean.reset_index(drop=True)

# Check result
print("Missing values in population:", population_clean["total_population"].isna().sum())
print("Cleaned dataset shape:", population_clean.shape)
population_clean.head()

Missing values in population: 0
Cleaned dataset shape: (3222, 3)


,county_fips,county_name,total_population
0,01001,"Autauga County, Alabama",59947
1,01003,"Baldwin County, Alabama",246989
2,01005,"Barbour County, Alabama",24643
3,01007,"Bibb County, Alabama",22130
4,01009,"Blount County, Alabama",59518


## Load Income Dataset (ACS Table B19013)

This section loads the median household income dataset from the American Community Survey (ACS).

The B19013 table provides median household income for each U.S. county. This is a key indicator of economic strength and will play an important role in calculating the economic risk score.

In [8]:
# -------------------------------------------------------
# Load & Inspect Income Dataset (B19013)
# -------------------------------------------------------

# Load income dataset
income_raw = pd.read_csv("ACSDT5Y2024.B19013-Data.csv")

# Check dataset shape
print("Raw dataset shape:", income_raw.shape)

# Preview data
income_raw.head()

Raw dataset shape: (3223, 5)


,GEO_ID,NAME,B19013_001E,B19013_001M,Unnamed: 4
0,Geography,Geographic Area Name,Estimate!!Median household income in the past ...,Margin of Error!!Median household income in th...,NaN
1,0500000US01001,"Autauga County, Alabama",72481,5640,NaN
2,0500000US01003,"Baldwin County, Alabama",78775,2602,NaN
3,0500000US01005,"Barbour County, Alabama",46042,5089,NaN
4,0500000US01007,"Bibb County, Alabama",52541,8950,NaN


## Clean Income Dataset

The raw income dataset is cleaned to keep only the information needed for the project.

In this step:
- the extra header row is removed
- the county FIPS code is extracted for dataset merging
- median household income is converted to numeric format
- only the required columns are retained

This creates a clean county-level income dataset ready to be merged with the other ACS tables.

In [10]:
# -------------------------------------------------------
# Clean Income Dataset
# -------------------------------------------------------

# Remove the first row containing extra header information
income_df = income_raw.iloc[1:].copy()

# Extract 5-digit county FIPS code from GEO_ID
income_df["county_fips"] = income_df["GEO_ID"].str[-5:]

# Convert median household income to numeric
income_df["median_household_income"] = pd.to_numeric(
    income_df["B19013_001E"],
    errors="coerce"
)

# Handle missing values using median
income_df["median_household_income"] = income_df["median_household_income"].fillna(
    income_df["median_household_income"].median()
)

# Keep only relevant columns
income_clean = income_df[
    ["county_fips", "median_household_income"]
].reset_index(drop=True)

# Check results
print("Missing income values after cleaning:", income_clean["median_household_income"].isna().sum())
print("Cleaned dataset shape:", income_clean.shape)
income_clean.head()

Missing income values after cleaning: 0
Cleaned dataset shape: (3222, 2)


,county_fips,median_household_income
0,01001,72481.0
1,01003,78775.0
2,01005,46042.0
3,01007,52541.0
4,01009,64190.0


## Load Poverty Dataset (ACS Table B17001)

This section loads the poverty dataset from the American Community Survey (ACS).

The B17001 table provides information about poverty status at the county level. This variable is important for measuring economic hardship and will be used as a key indicator in calculating the economic risk score.

In [12]:
# -------------------------------------------------------
# Load & Inspect Poverty Dataset (B17001)
# -------------------------------------------------------

# Load poverty dataset
poverty_raw = pd.read_csv("ACSDT5Y2024.B17001-Data.csv")

# Check dataset shape
print("Raw dataset shape:", poverty_raw.shape)

# Preview first few rows
poverty_raw.head()

Raw dataset shape: (3223, 121)


,GEO_ID,NAME,B17001_001E,B17001_001M,B17001_002E,B17001_002M,B17001_003E,B17001_003M,B17001_004E,B17001_004M,B17001_005E,B17001_005M,B17001_006E,B17001_006M,B17001_007E,B17001_007M,B17001_008E,B17001_008M,B17001_009E,B17001_009M,B17001_010E,B17001_010M,B17001_011E,B17001_011M,B17001_012E,...,B17001_048E,B17001_048M,B17001_049E,B17001_049M,B17001_050E,B17001_050M,B17001_051E,B17001_051M,B17001_052E,B17001_052M,B17001_053E,B17001_053M,B17001_054E,B17001_054M,B17001_055E,B17001_055M,B17001_056E,B17001_056M,B17001_057E,B17001_057M,B17001_058E,B17001_058M,B17001_059E,B17001_059M,Unnamed: 120
0,Geography,Geographic Area Name,Estimate!!Total:,Margin of Error!!Total:,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,Estimate!!Total:!!Income in the past 12 months...,Margin of Error!!Total:!!Income in the past 12...,NaN
1,0500000US01001,"Autauga County, Alabama",59480,79,6715,1269,3033,669,286,149,114,85,367,164,170,87,76,77,82,78,285,163,171,162,371,...,124,72,1823,296,1256,262,300,102,935,160,1911,208,3478,270,3733,210,3570,276,3453,220,2747,119,2203,116,NaN
2,0500000US01003,"Baldwin County, Alabama",243473,332,24573,2537,10383,1323,1230,270,438,232,1135,363,798,298,39,46,339,182,698,273,1049,420,1026,...,1369,368,6679,660,4924,583,1692,399,2234,378,7031,525,12118,437,13575,420,14762,441,16118,615,15555,270,10186,346,NaN
3,0500000US01005,"Barbour County, Alabama",21583,80,4615,693,1893,334,265,78,69,58,139,81,98,84,0,25,31,27,195,157,457,143,123,...,109,94,300,131,339,123,55,40,299,70,595,120,939,133,1144,96,1025,111,1267,134,1302,77,1011,102,NaN
4,0500000US01007,"Bibb County, Alabama",20488,260,4601,693,2104,428,193,144,0,25,259,126,234,139,69,50,79,50,272,115,261,128,169,...,42,42,409,138,256,97,52,44,155,73,419,100,1052,149,972,109,1071,105,1042,163,965,103,846,93,NaN


## Clean Poverty Dataset and Compute Poverty Rate

The raw poverty dataset is cleaned and transformed to create a meaningful poverty indicator.

In this step:
- the extra header row is removed
- the county FIPS code is extracted for merging
- relevant columns are converted to numeric format
- the poverty rate is calculated as the proportion of people below the poverty line

The poverty rate provides a standardized measure of economic hardship across counties.

In [14]:
# -------------------------------------------------------
# Clean Poverty Dataset + Compute Poverty Rate
# -------------------------------------------------------

poverty_df = poverty_raw.iloc[1:].copy()

# Extract county FIPS
poverty_df["county_fips"] = poverty_df["GEO_ID"].str[-5:]

# Convert required columns to numeric
poverty_df["poverty_population_total"] = pd.to_numeric(poverty_df["B17001_001E"], errors="coerce")
poverty_df["population_below_poverty"] = pd.to_numeric(poverty_df["B17001_002E"], errors="coerce")

# Compute poverty rate
poverty_df["poverty_rate"] = poverty_df["population_below_poverty"] / poverty_df["poverty_population_total"]

# Keep only final columns
poverty_clean = poverty_df[
    ["county_fips", "poverty_rate"]
].reset_index(drop=True)
min_val = poverty_clean["poverty_rate"].min()
max_val = poverty_clean["poverty_rate"].max()

print("Missing poverty rate values:", poverty_clean["poverty_rate"].isna().sum())
print(f"Poverty rate range: {min_val:.2f} - {max_val:.2f}")
print("Cleaned shape:", poverty_clean.shape)
poverty_clean.head()

Missing poverty rate values: 0
Poverty rate range: 0.01 - 0.64
Cleaned shape: (3222, 2)


,county_fips,poverty_rate
0,01001,0.112895
1,01003,0.100927
2,01005,0.213826
3,01007,0.224570
4,01009,0.128601


## Load Employment Dataset (ACS Table B23025)

This section loads the employment dataset from the American Community Survey (ACS).

The B23025 table provides information about labor force participation, employment, and unemployment at the county level. The unemployment rate is an important indicator of economic health and will be used in calculating the economic risk score.

In [16]:
# -------------------------------------------------------
# Load & Inspect Employment Dataset (B23025)
# -------------------------------------------------------

# Load employment dataset
employment_raw = pd.read_csv("ACSDT5Y2024.B23025-Data.csv")

# Check dataset shape
print("Raw dataset shape:", employment_raw.shape)

# Preview first few rows
employment_raw.head()

Raw dataset shape: (3223, 17)


,GEO_ID,NAME,B23025_001E,B23025_001M,B23025_002E,B23025_002M,B23025_003E,B23025_003M,B23025_004E,B23025_004M,B23025_005E,B23025_005M,B23025_006E,B23025_006M,B23025_007E,B23025_007M,Unnamed: 16
0,Geography,Geographic Area Name,Estimate!!Total:,Margin of Error!!Total:,Estimate!!Total:!!In labor force:,Margin of Error!!Total:!!In labor force:,Estimate!!Total:!!In labor force:!!Civilian la...,Margin of Error!!Total:!!In labor force:!!Civi...,Estimate!!Total:!!In labor force:!!Civilian la...,Margin of Error!!Total:!!In labor force:!!Civi...,Estimate!!Total:!!In labor force:!!Civilian la...,Margin of Error!!Total:!!In labor force:!!Civi...,Estimate!!Total:!!In labor force:!!Armed Forces,Margin of Error!!Total:!!In labor force:!!Arme...,Estimate!!Total:!!Not in labor force,Margin of Error!!Total:!!Not in labor force,NaN
1,0500000US01001,"Autauga County, Alabama",47821,213,27992,908,26763,1001,26122,1010,641,198,1229,371,19829,932,NaN
2,0500000US01003,"Baldwin County, Alabama",200551,479,118112,1932,117463,1962,113907,2026,3556,589,649,250,82439,2012,NaN
3,0500000US01005,"Barbour County, Alabama",20333,108,9275,498,9256,496,8535,480,721,221,19,27,11058,498,NaN
4,0500000US01007,"Bibb County, Alabama",18230,99,9064,463,9064,463,7969,509,1095,276,0,25,9166,446,NaN


## Clean Employment Dataset and Compute Unemployment Rate

The employment dataset is cleaned and transformed to create a meaningful unemployment indicator.

In this step:
- the extra header row is removed
- the county FIPS code is extracted for merging
- labor force and unemployed counts are converted to numeric format
- the unemployment rate is calculated as the proportion of unemployed individuals in the labor force

The unemployment rate provides a standardized measure of labor market conditions across counties.

In [18]:
# -------------------------------------------------------
# Clean Employment Dataset + Compute Unemployment Rate
# -------------------------------------------------------

# Remove extra header row
employment_df = employment_raw.iloc[1:].copy()

# Extract county FIPS code
employment_df["county_fips"] = employment_df["GEO_ID"].str[-5:]

# Convert required columns to numeric
# B23025_002E = Labor force
# B23025_005E = Unemployed
employment_df["labor_force"] = pd.to_numeric(
    employment_df["B23025_002E"], errors="coerce"
)

employment_df["unemployed"] = pd.to_numeric(
    employment_df["B23025_005E"], errors="coerce"
)

# Compute unemployment rate
employment_df["unemployment_rate"] = (
    employment_df["unemployed"] / employment_df["labor_force"]
)

# Keep only required columns
employment_clean = employment_df[
    ["county_fips", "unemployment_rate"]
].reset_index(drop=True)

# Preview cleaned dataset
print("Missing unemployment rate values:", employment_clean["unemployment_rate"].isna().sum())
print(f"Unemployment rate range: {employment_clean['unemployment_rate'].min():.2f} - {employment_clean['unemployment_rate'].max():.2f}")
print("Cleaned dataset shape:", employment_clean.shape)
employment_clean.head()

Missing unemployment rate values: 0
Unemployment rate range: 0.00 - 0.29
Cleaned dataset shape: (3222, 2)


,county_fips,unemployment_rate
0,01001,0.022899
1,01003,0.030107
2,01005,0.077736
3,01007,0.120808
4,01009,0.049971


## Load Housing Tenure Dataset (ACS Table B25003)

This section loads the housing tenure dataset from the American Community Survey (ACS).

The B25003 table provides information on owner-occupied and renter-occupied housing units at the county level. Housing tenure is useful for understanding residential stability, and homeownership rate will be used as one of the socioeconomic indicators in the analysis.

In [20]:
# -------------------------------------------------------
# Load & Inspect Housing Tenure Dataset (B25003)
# -------------------------------------------------------

# Load housing tenure dataset
housing_raw = pd.read_csv("ACSDT5Y2024.B25003-Data.csv")

# Check dataset shape
print("Raw dataset shape:", housing_raw.shape)

# Preview first few rows
housing_raw.head()

Raw dataset shape: (3223, 9)


,GEO_ID,NAME,B25003_001E,B25003_001M,B25003_002E,B25003_002M,B25003_003E,B25003_003M,Unnamed: 8
0,Geography,Geographic Area Name,Estimate!!Total:,Margin of Error!!Total:,Estimate!!Total:!!Owner occupied,Margin of Error!!Total:!!Owner occupied,Estimate!!Total:!!Renter occupied,Margin of Error!!Total:!!Renter occupied,NaN
1,0500000US01001,"Autauga County, Alabama",22917,435,17660,656,5257,514,NaN
2,0500000US01003,"Baldwin County, Alabama",98574,1224,76497,1493,22077,1398,NaN
3,0500000US01005,"Barbour County, Alabama",9128,308,6229,330,2899,312,NaN
4,0500000US01007,"Bibb County, Alabama",7704,283,6100,340,1604,311,NaN


## Clean Housing Dataset and Compute Housing Rates

The housing dataset is cleaned and transformed to create meaningful housing indicators.

In this step:
- the extra header row is removed
- the county FIPS code is extracted for merging
- housing-related counts are converted to numeric format
- homeownership rate and renter rate are calculated

These rates provide standardized measures of housing stability and will be used as indicators in the economic risk analysis.

In [22]:
# -------------------------------------------------------
# Clean Housing Dataset + Compute Housing Rates
# -------------------------------------------------------

# Remove extra header row
housing_df = housing_raw.iloc[1:].copy()

# Extract county FIPS code
housing_df["county_fips"] = housing_df["GEO_ID"].str[-5:]

# Convert required columns to numeric
# B25003_001E = Total housing units
# B25003_002E = Owner-occupied
# B25003_003E = Renter-occupied
housing_df["total_housing_units"] = pd.to_numeric(
    housing_df["B25003_001E"], errors="coerce"
)

housing_df["owner_occupied"] = pd.to_numeric(
    housing_df["B25003_002E"], errors="coerce"
)

housing_df["renter_occupied"] = pd.to_numeric(
    housing_df["B25003_003E"], errors="coerce"
)

# Compute housing rates
housing_df["homeownership_rate"] = (
    housing_df["owner_occupied"] / housing_df["total_housing_units"]
)

housing_df["renter_rate"] = (
    housing_df["renter_occupied"] / housing_df["total_housing_units"]
)

# Keep only relevant columns
housing_clean = housing_df[
    ["county_fips", "homeownership_rate", "renter_rate"]
].reset_index(drop=True)

# Preview cleaned dataset
print("Missing homeownership values:", housing_clean["homeownership_rate"].isna().sum())
print(f"Homeownership rate range: {housing_clean['homeownership_rate'].min():.2f} - {housing_clean['homeownership_rate'].max():.2f}")
print("Missing renter values:", housing_clean["renter_rate"].isna().sum())
print(f"Renter rate range: {housing_clean['renter_rate'].min():.2f} - {housing_clean['renter_rate'].max():.2f}")
print("Cleaned dataset shape:", housing_clean.shape)
housing_clean.head()

Missing homeownership values: 0
Homeownership rate range: 0.00 - 0.96
Missing renter values: 0
Renter rate range: 0.04 - 1.00
Cleaned dataset shape: (3222, 3)


,county_fips,homeownership_rate,renter_rate
0,01001,0.770607,0.229393
1,01003,0.776036,0.223964
2,01005,0.682406,0.317594
3,01007,0.791796,0.208204
4,01009,0.809829,0.190171


## Load Education Dataset (ACS Table B15003)

This section loads the education dataset from the American Community Survey (ACS).

The B15003 table provides detailed information about educational attainment at the county level. In this project, it will be used to calculate the percentage of adults with a bachelor’s degree or higher, which is an important indicator of economic opportunity and long-term stability.

In [24]:
# -------------------------------------------------------
# Load & Inspect Education Dataset (B15003)
# -------------------------------------------------------

# Load education dataset
education_raw = pd.read_csv("ACSDT5Y2024.B15003-Data.csv")

# Check dataset shape
print("Raw dataset shape:", education_raw.shape)

# Preview first few rows
education_raw.head()

Raw dataset shape: (3223, 53)


,GEO_ID,NAME,B15003_001E,B15003_001M,B15003_002E,B15003_002M,B15003_003E,B15003_003M,B15003_004E,B15003_004M,B15003_005E,B15003_005M,B15003_006E,B15003_006M,B15003_007E,B15003_007M,B15003_008E,B15003_008M,B15003_009E,B15003_009M,B15003_010E,B15003_010M,B15003_011E,B15003_011M,B15003_012E,...,B15003_014E,B15003_014M,B15003_015E,B15003_015M,B15003_016E,B15003_016M,B15003_017E,B15003_017M,B15003_018E,B15003_018M,B15003_019E,B15003_019M,B15003_020E,B15003_020M,B15003_021E,B15003_021M,B15003_022E,B15003_022M,B15003_023E,B15003_023M,B15003_024E,B15003_024M,B15003_025E,B15003_025M,Unnamed: 52
0,Geography,Geographic Area Name,Estimate!!Total:,Margin of Error!!Total:,Estimate!!Total:!!No schooling completed,Margin of Error!!Total:!!No schooling completed,Estimate!!Total:!!Nursery school,Margin of Error!!Total:!!Nursery school,Estimate!!Total:!!Kindergarten,Margin of Error!!Total:!!Kindergarten,Estimate!!Total:!!1st grade,Margin of Error!!Total:!!1st grade,Estimate!!Total:!!2nd grade,Margin of Error!!Total:!!2nd grade,Estimate!!Total:!!3rd grade,Margin of Error!!Total:!!3rd grade,Estimate!!Total:!!4th grade,Margin of Error!!Total:!!4th grade,Estimate!!Total:!!5th grade,Margin of Error!!Total:!!5th grade,Estimate!!Total:!!6th grade,Margin of Error!!Total:!!6th grade,Estimate!!Total:!!7th grade,Margin of Error!!Total:!!7th grade,Estimate!!Total:!!8th grade,...,Estimate!!Total:!!10th grade,Margin of Error!!Total:!!10th grade,Estimate!!Total:!!11th grade,Margin of Error!!Total:!!11th grade,"Estimate!!Total:!!12th grade, no diploma","Margin of Error!!Total:!!12th grade, no diploma",Estimate!!Total:!!Regular high school diploma,Margin of Error!!Total:!!Regular high school d...,Estimate!!Total:!!GED or alternative credential,Margin of Error!!Total:!!GED or alternative cr...,"Estimate!!Total:!!Some college, less than 1 year","Margin of Error!!Total:!!Some college, less th...","Estimate!!Total:!!Some college, 1 or more year...","Margin of Error!!Total:!!Some college, 1 or mo...",Estimate!!Total:!!Associate's degree,Margin of Error!!Total:!!Associate's degree,Estimate!!Total:!!Bachelor's degree,Margin of Error!!Total:!!Bachelor's degree,Estimate!!Total:!!Master's degree,Margin of Error!!Total:!!Master's degree,Estimate!!Total:!!Professional school degree,Margin of Error!!Total:!!Professional school d...,Estimate!!Total:!!Doctorate degree,Margin of Error!!Total:!!Doctorate degree,NaN
1,0500000US01001,"Autauga County, Alabama",41392,122,286,154,24,28,0,31,0,31,0,31,0,31,8,17,19,29,49,43,9,13,224,...,982,277,807,237,689,217,11483,822,2472,404,2699,378,6080,685,3004,500,6378,592,4764,576,598,250,321,133,NaN
2,0500000US01003,"Baldwin County, Alabama",177628,237,1645,381,127,100,98,111,63,83,52,61,0,31,61,54,43,44,869,404,332,245,830,...,2838,566,2284,566,3702,743,39036,1710,9044,1182,13284,1149,24549,1275,17325,1031,37938,1824,15646,1168,4035,665,2223,462,NaN
3,0500000US01005,"Barbour County, Alabama",17615,76,207,77,73,77,37,46,68,59,0,25,96,78,20,23,43,37,124,69,162,88,401,...,1072,241,661,164,518,122,5286,337,1385,271,1044,218,2424,266,1752,239,1139,208,497,126,181,108,55,45,NaN
4,0500000US01007,"Bibb County, Alabama",15965,206,192,127,0,25,4,11,0,25,0,25,11,19,0,25,34,44,140,92,84,53,444,...,438,152,608,242,454,180,5859,517,920,221,952,233,2293,392,1237,246,1173,267,548,191,63,38,88,54,NaN


## Clean Education Dataset and Compute Bachelor's or Higher Percentage

The education dataset is cleaned and transformed to create a meaningful education indicator.

In this step:
- the extra header row is removed
- the county FIPS code is extracted for merging
- relevant education columns are converted to numeric format
- the percentage of adults with a bachelor's degree or higher is calculated

This variable represents the level of higher education attainment in each county and is an important indicator of long-term economic stability.

In [26]:
# -------------------------------------------------------
# Clean Education Dataset + Bachelor's or Higher %
# -------------------------------------------------------

# Remove extra header row
education_df = education_raw.iloc[1:].copy()

# Extract county FIPS code
education_df["county_fips"] = education_df["GEO_ID"].str[-5:]

# Columns used for calculation
# B15003_001E = Total population (age 25+)
# B15003_022E–025E = Bachelor's and higher degrees
cols = [
    "B15003_001E",
    "B15003_022E",
    "B15003_023E",
    "B15003_024E",
    "B15003_025E"
]

# Convert columns to numeric
for c in cols:
    education_df[c] = pd.to_numeric(education_df[c], errors="coerce")

# Compute percentage of population with bachelor's or higher
education_df["bachelors_or_higher_pct"] = (
    education_df["B15003_022E"] +
    education_df["B15003_023E"] +
    education_df["B15003_024E"] +
    education_df["B15003_025E"]
) / education_df["B15003_001E"]

# Keep only relevant columns
education_clean = education_df[
    ["county_fips", "bachelors_or_higher_pct"]
].reset_index(drop=True)

# Preview cleaned dataset
print("Missing education values:", education_clean["bachelors_or_higher_pct"].isna().sum())
print(f"Education rate range: {education_clean['bachelors_or_higher_pct'].min():.2f} - {education_clean['bachelors_or_higher_pct'].max():.2f}")
print("Cleaned dataset shape:", education_clean.shape)
education_clean.head()

Missing education values: 0
Education rate range: 0.00 - 0.81
Cleaned dataset shape: (3222, 2)


,county_fips,bachelors_or_higher_pct
0,01001,0.291385
1,01003,0.336895
2,01005,0.106273
3,01007,0.117256
4,01009,0.158182


## Merge Cleaned Datasets into Master County Dataset

In this step, all cleaned datasets are merged into a single master dataset at the county level.

Each dataset contains a different socioeconomic indicator:
- population
- income
- poverty rate
- unemployment rate
- housing indicators
- education level

All datasets are merged using the county FIPS code to ensure consistency. The result is a unified dataset containing all relevant features for each county.

In [28]:
# -------------------------------------------------------
# Merge All Clean Tables (Master County Dataset)
# -------------------------------------------------------

# Start with population dataset as base
df_county = population_clean.copy()

# Merge all other datasets using county FIPS
df_county = df_county.merge(income_clean, on="county_fips", how="left")
df_county = df_county.merge(poverty_clean, on="county_fips", how="left")
df_county = df_county.merge(employment_clean, on="county_fips", how="left")
df_county = df_county.merge(housing_clean, on="county_fips", how="left")
df_county = df_county.merge(education_clean, on="county_fips", how="left")

# Check final dataset
print("\nMissing values per column:")
print(df_county.isna().sum())
print("Final merged dataset shape:", df_county.shape)
df_county.head()


Missing values per column:
county_fips                0
county_name                0
total_population           0
median_household_income    0
poverty_rate               0
unemployment_rate          0
homeownership_rate         0
renter_rate                0
bachelors_or_higher_pct    0
dtype: int64
Final merged dataset shape: (3222, 9)


,county_fips,county_name,total_population,median_household_income,poverty_rate,unemployment_rate,homeownership_rate,renter_rate,bachelors_or_higher_pct
0,01001,"Autauga County, Alabama",59947,72481.0,0.112895,0.022899,0.770607,0.229393,0.291385
1,01003,"Baldwin County, Alabama",246989,78775.0,0.100927,0.030107,0.776036,0.223964,0.336895
2,01005,"Barbour County, Alabama",24643,46042.0,0.213826,0.077736,0.682406,0.317594,0.106273
3,01007,"Bibb County, Alabama",22130,52541.0,0.224570,0.120808,0.791796,0.208204,0.117256
4,01009,"Blount County, Alabama",59518,64190.0,0.128601,0.049971,0.809829,0.190171,0.158182


## Final Data Validation

Before proceeding to analysis, the dataset is validated to ensure data quality and consistency.

The following checks are performed:
- Identification of duplicate county records
- Calculation of missing values as a percentage for each column

These checks help confirm that the dataset is clean and ready for analysis.

In [39]:
# -------------------------------------------------------
# Final Data Validation
# -------------------------------------------------------

# Check for duplicate counties
duplicate_counties = df_county["county_fips"].duplicated().sum()
print("Duplicate county_fips:", duplicate_counties)

# Check missing values (percentage)
missing_pct = (df_county.isna().mean() * 100).round(2)
print("\nMissing values (%) by column:")
print(missing_pct)

# Dataset summary
print("\nDataset summary:")
print(df_county.describe())

Duplicate county_fips: 0

Missing values (%) by column:
county_fips                0.0
county_name                0.0
total_population           0.0
median_household_income    0.0
poverty_rate               0.0
unemployment_rate          0.0
homeownership_rate         0.0
renter_rate                0.0
bachelors_or_higher_pct    0.0
dtype: float64

Dataset summary:
       total_population  median_household_income  poverty_rate  unemployment_rate  homeownership_rate  renter_rate  \
count      3.222000e+03              3222.000000   3222.000000        3222.000000         3222.000000  3222.000000   
mean       1.049525e+05             66934.803228      0.149401           0.047796            0.731574     0.268426   
std        3.312740e+05             18777.795318      0.075273           0.026435            0.085715     0.085715   
min        3.300000e+01             16314.000000      0.014159           0.000000            0.000000     0.039112   
25%        1.098325e+04             55834.

## Save Final Cleaned Dataset

After completing data cleaning, feature engineering, merging, and validation, the final dataset is saved for use in subsequent analysis and modeling.

This dataset contains all key socioeconomic indicators at the county level and will be used in the exploratory data analysis (EDA) and modeling notebooks.

In [44]:
# -------------------------------------------------------
# Save Final Cleaned Dataset
# -------------------------------------------------------

# Define output file path
output_path = "data/processed/county_master.csv"

# Save dataset
df_county.to_csv(output_path, index=False)

# Confirm save
print(f"Cleaned county dataset saved to: {output_path}")
print("Final dataset shape:", df_county.shape)

Cleaned county dataset saved to: data/processed/county_master.csv
Final dataset shape: (3222, 9)
